# nb_lakehouse_maintenance — act on the health report (dry-run by default)
**Platform pattern:** destructive operations are metadata-driven, dry-run first, and logged.
This notebook reads the latest health report from the audit notebook and executes only the flagged
actions: `OPTIMIZE` for small-file tables, `REORG … APPLY (PURGE)` where deletion vectors have
accumulated, `VACUUM` past retention. `DRY_RUN=True` prints the exact plan without touching anything —
flip it in the pipeline parameter, not in the code.

In [1]:
NOTEBOOK_NAME = "nb_lakehouse_maintenance"
TABLES_ROOT   = "/tmp/fabric_kit_warehouse"
REPORT_TABLE  = f"{TABLES_ROOT}/_ops/table_health"
LOG_TABLE     = f"{TABLES_ROOT}/_ops/maintenance_log"
DRY_RUN       = False        # ship default True in production; False here to demonstrate execution
VACUUM_RETAIN_HOURS = 168    # never below your time-travel / CDF consumer window

In [2]:
# --- Session: Fabric is the default target -------------------------------------
# In Fabric you do NOT create a Spark session. The Livy layer starts it before your first
# cell runs, and `spark` (plus `sc`, `notebookutils`) are already bound. Calling
# SparkSession.builder there is at best a no-op via getOrCreate() and at worst misleading:
# master(), Delta wiring and executor shape are all decided by the Environment/pool, not here.
#
# Session-start settings belong in a %%configure -f cell ABOVE this one, or in the
# Environment's Spark properties. Only runtime-mutable keys can be set from code.
try:
    spark                      # noqa: F821  <- Fabric (and any live session): already provided
    IN_FABRIC = True
except NameError:
    # Local/dev fallback ONLY. Never runs in Fabric.
    IN_FABRIC = False
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip
    _b = (SparkSession.builder.appName(NOTEBOOK_NAME).master("local[4]")
          .config("spark.driver.memory", "2g")
          .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
          .config("spark.sql.catalog.spark_catalog",
                  "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(_b).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(("Fabric session (provided)" if IN_FABRIC else "local session (dev fallback)"),
      "| Spark", spark.version)

# --- session bootstrap (identical in every kit notebook; Fabric supplies `spark`) ---
import os, sys, json, time
from datetime import datetime, timezone

def get_session():
    try:
        return spark  # noqa: F821  (Fabric / existing session)
    except NameError:
        from pyspark.sql import SparkSession
        from delta import configure_spark_with_delta_pip
        b = (SparkSession.builder.appName(NOTEBOOK_NAME).master("local[4]")
             .config("spark.driver.memory", "2g")
             .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
             .config("spark.sql.catalog.spark_catalog",
                     "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        return configure_spark_with_delta_pip(b).getOrCreate()

spark = get_session()
spark.sparkContext.setLogLevel("ERROR")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
APP_ID = spark.sparkContext.applicationId
print(f"{NOTEBOOK_NAME} | run {RUN_ID} | app {APP_ID} | Spark {spark.version}")

26/08/04 12:17:03 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/04 12:17:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-84e18c8b-f280-4442-822b-2e57bccd34c9;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central


	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 257ms :: artifacts dl 18ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-84e18c8b-f280-4442-822b-2e57bccd34c9
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/11ms)


26/08/04 12:17:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


local session (dev fallback) | Spark 3.5.1
nb_lakehouse_maintenance | run 20260804T121707Z | app local-1785845826485 | Spark 3.5.1


In [3]:
from pyspark.sql import functions as F
report = spark.read.format("delta").load(REPORT_TABLE)
latest_run = report.agg(F.max("run_id")).collect()[0][0]
todo = report.where((F.col("run_id") == latest_run) & (F.col("flags") != "OK")).collect()
print(f"acting on audit {latest_run}: {len(todo)} flagged table(s)")

log = []
for t in todo:
    for flag in t["flags"].split(","):
        action = {"COMPACT": f"OPTIMIZE delta.`{t['path']}`",
                  "NEVER_OPTIMIZED": f"OPTIMIZE delta.`{t['path']}`",
                  "PURGE_DV": f"REORG TABLE delta.`{t['path']}` APPLY (PURGE)",
                  "VACUUM_OVERDUE": f"VACUUM delta.`{t['path']}` RETAIN {VACUUM_RETAIN_HOURS} HOURS"}.get(flag)
        if not action: continue
        entry = {"run_id": RUN_ID, "audit_run": latest_run, "table_name": t["table_name"],
                 "flag": flag, "sql": action, "dry_run": DRY_RUN, "status": "PLANNED",
                 "app_id": APP_ID, "started": datetime.now(timezone.utc).isoformat()}
        if DRY_RUN:
            print("[DRY RUN]", action)
        else:
            t0 = time.time()
            try:
                spark.sql(action)
                entry.update(status="OK", secs=round(time.time()-t0, 1))
                print("[DONE]", action, f"({entry['secs']}s)")
            except Exception as ex:
                entry.update(status="FAILED", error=str(ex)[:300])
                print("[FAILED]", action, "->", str(ex)[:120])
        log.append(entry)

if log:
    spark.createDataFrame(log).write.format("delta").mode("append").option("mergeSchema","true").save(LOG_TABLE)
    assert all(e["status"] != "FAILED" for e in log), "maintenance had failures - see log table"
print(f"maintenance complete: {len(log)} action(s), dry_run={DRY_RUN}")

acting on audit 20260804T120729Z: 0 flagged table(s)
maintenance complete: 0 action(s), dry_run=False


In [4]:
# Post-check: re-derive the one metric OPTIMIZE should have moved.
for t in todo:
    if "COMPACT" in t["flags"] and not DRY_RUN:
        d = spark.sql(f"DESCRIBE DETAIL delta.`{t['path']}`").collect()[0]
        print(f"{t['table_name']}: files {t['num_files']} -> {d['numFiles']}")
        assert d["numFiles"] < t["num_files"], "compaction did not reduce file count"
spark.stop() if "local" in spark.sparkContext.master else None